# ▶ 유튜브 영상 추출 / 다운로드
* 아래의 아래 셀을 실행해 yt-dlp와 ffmpeg를 install 합니다.
* 매인 셀을 실행해서 다운받을 유튜브 주소를 복사/붙여넣기하여 실행합니다.
* 이후 원하는 화질을 선택하면 PC로 자동 다운로드 됩니다.

In [ ]:
!pip install -U yt-dlp
!apt-get update
!apt-get install ffmpeg -y

In [ ]:
import yt_dlp
from google.colab import files

# ==============================
# 파일 크기 변환
# ==============================

def format_size(size):

    if size is None:
        return "알 수 없음"

    for unit in ["B","KB","MB","GB"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} TB"


# ==============================
# 다운로드 진행률
# ==============================

def progress_hook(d):

    if d["status"] == "downloading":

        downloaded = d.get("downloaded_bytes",0)
        total = d.get("total_bytes") or d.get("total_bytes_estimate")

        if total:
            percent = downloaded / total * 100
            print(f"\r다운로드 진행률: {percent:.1f}% ({format_size(downloaded)} / {format_size(total)})", end="")

    elif d["status"] == "finished":
        print("\n다운로드 완료. 병합 중...")


# ==============================
# 영상 정보 가져오기
# ==============================

def get_video_info(url):

    ydl_opts = {"quiet": True}

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    title = info["title"]

    print("\n제목:", title)

    duration = info.get("duration",0)
    print("영상 길이:", duration//60,"분",duration%60,"초")

    heights = {}

    for f in info["formats"]:
        if f.get("height") and f.get("vcodec") != "none":

            h = f["height"]
            size = f.get("filesize") or f.get("filesize_approx")

            if h not in heights:
                heights[h] = size
            else:
                if size and heights[h] and size > heights[h]:
                    heights[h] = size

    heights = dict(sorted(heights.items(), reverse=True))

    print("\n=== 선택 가능한 화질 ===")

    height_list = list(heights.keys())

    for i,h in enumerate(height_list):

        size = format_size(heights[h])
        print(i,":",f"{h}p  (약 {size})")

    return height_list, title


# ==============================
# 다운로드
# ==============================

def download_video(url, height):

    ydl_opts = {
        "format": f"bv*[height<={height}][vcodec^=avc1]+ba[ext=m4a]/b[ext=mp4]",
        "outtmpl": "/content/%(title)s.%(ext)s",
        "progress_hooks": [progress_hook],
        "merge_output_format": "mp4",
        "noplaylist": True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)

    return filename


# ==============================
# 실행
# ==============================

urls = input("유튜브 URL 입력 (여러개면 , 로 구분): ").split(",")

for url in urls:

    url = url.strip()

    print("\n영상 분석 중...")

    heights, title = get_video_info(url)

    choice = int(input("\n다운로드할 화질 번호 선택: "))

    selected_height = heights[choice]

    print(f"\n{selected_height}p 다운로드 시작...\n")

    file = download_video(url, selected_height)

    print("\n💻 PC로 다운로드 시작")

    files.download(file)

print('\n✅ Download')